In [1]:
import pandas as pd
import itertools
import requests

In [2]:
df_raw_imdb = pd.read_csv('raw_imdb.csv')
df_raw_imdb = df_raw_imdb[['primaryTitle', 'startYear', 'numVotes', 'averageRating', 'genres']]
df_raw_imdb

,primaryTitle,startYear,numVotes,averageRating,genres
0,Dante's Inferno,1911,4225,7.1,"Adventure,Drama,Fantasy"
1,Fantômas: In the Shadow of the Guillotine,1913,2746,6.9,"Crime,Drama"
2,Ingeborg Holm,1913,1623,7.0,Drama
3,Fantomas: The Man in Black,1913,1864,6.9,"Crime,Drama"
4,Cabiria,1914,4381,7.1,"Adventure,Drama,History"
...,...,...,...,...,...
18115,Mogul Mowgli,2020,3462,6.6,"Drama,Music"
18116,Min pappa Marianne,2020,2556,6.8,"Comedy,Drama"
18117,Kaithi,2019,52364,8.4,"Action,Crime,Thriller"
18118,Herself,2020,5398,7.0,Drama


In [3]:
genres_list = df_raw_imdb['genres'].dropna().str.split(',')
genres_set = set(itertools.chain.from_iterable(genres_list))
genres_set.add('')
genres_set

{'',
 'Action',
 'Adult',
 'Adventure',
 'Animation',
 'Biography',
 'Comedy',
 'Crime',
 'Documentary',
 'Drama',
 'Family',
 'Fantasy',
 'Film-Noir',
 'History',
 'Horror',
 'Music',
 'Musical',
 'Mystery',
 'News',
 'Romance',
 'Sci-Fi',
 'Sport',
 'Talk-Show',
 'Thriller',
 'War',
 'Western'}

In [4]:
len(genres_set)

26

In [5]:
import time

def get_tmdb_info(title, year, retries=3, backoff=2):
    api_key = '7d50f2eef53ed745dbc5a731c72dfa36'
    search_url = 'https://api.themoviedb.org/3/search/movie'
    params = {
        'api_key': api_key,
        'query': title,
        'year': year,
        'language': 'zh-TW'
    }
    for attempt in range(retries):
        try:
            resp = requests.get(search_url, params=params, timeout=10).json()
            if not resp.get('results'):
                return '', '', ''
            movie_id = resp['results'][0]['id']
            taiwan_title = resp['results'][0].get('title', '')
            poster_path = resp['results'][0].get('poster_path', '')
            poster_path = f"https://image.tmdb.org/t/p/w500{poster_path}" if poster_path else ''

            video_url = f'https://api.themoviedb.org/3/movie/{movie_id}/videos'
            video_resp = requests.get(video_url, params={'api_key': api_key}, timeout=10).json()
            trailer_link = ''
            for v in video_resp.get('results', []):
                if v['type'] == 'Trailer' and v['site'] == 'YouTube':
                    trailer_link = f"https://www.youtube.com/watch?v={v['key']}"
                    break
            return taiwan_title, trailer_link, poster_path
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(backoff * (attempt + 1))
            else:
                print(f"  [WARN] Failed to fetch TMDB info for '{title}' ({year}): {e}")
                return '', '', ''


In [6]:
import os

output_dir = 'all genres'
os.makedirs(output_dir, exist_ok=True)

for genre in genres_set:
    if genre == '':
        df_genre = df_raw_imdb[df_raw_imdb['genres'].isnull()]
        filename = os.path.join(output_dir, 'empty.csv')
    else:
        df_genre = df_raw_imdb[df_raw_imdb['genres'].fillna('').str.contains(genre)]
        filename = os.path.join(output_dir, f'{genre}.csv')
    percentile_60 = df_genre['numVotes'].quantile(0.6)
    threshold = min(percentile_60, 10000)
    df_genre_filtered = df_genre[df_genre['numVotes'] > threshold]
    print(f'Processing genre: {genre}')
    print(f'numVotes threshold: {threshold}')
    print(len(df_genre_filtered))
    df_genre_filtered.drop(columns=['genres'], inplace=True)
    df_genre_filtered = df_genre_filtered[df_genre_filtered["averageRating"]>=6.5]
    df_genre_filtered.to_csv(filename, index=False)


Processing genre: 
numVotes threshold: 2670.0
8
Processing genre: Sport
numVotes threshold: 9314.199999999999
174
Processing genre: News
numVotes threshold: 4638.0
8
Processing genre: Romance
numVotes threshold: 8304.599999999999
1315
Processing genre: Western
numVotes threshold: 10000
104
Processing genre: Thriller
numVotes threshold: 10000
929
Processing genre: Action
numVotes threshold: 10000
1310
Processing genre: Comedy
numVotes threshold: 9788.2
2164
Processing genre: Talk-Show
numVotes threshold: 1908.0
0
Processing genre: Horror
numVotes threshold: 10000
417
Processing genre: Fantasy
numVotes threshold: 10000
367
Processing genre: Family
numVotes threshold: 8733.4
279
Processing genre: Film-Noir
numVotes threshold: 5583.6
120
Processing genre: Mystery
numVotes threshold: 10000
614
Processing genre: History
numVotes threshold: 8141.399999999999
438
Processing genre: Musical
numVotes threshold: 6316.0
162
Processing genre: Animation
numVotes threshold: 10000
397
Processing genre:

In [7]:
import os

output_dir = 'all genres'

with pd.ExcelWriter('all_genres.xlsx', engine='openpyxl') as writer:
    for genre in genres_set:
        if genre == '':
            genre_name = 'Empty'
            df_genre = pd.read_csv(os.path.join(output_dir, 'empty.csv'))
        else:
            genre_name = genre
            df_genre = pd.read_csv(os.path.join(output_dir, f'{genre}.csv'))
        
        percentile_90 = df_genre['averageRating'].quantile(0.9)
        print(f'Processing genre: {genre_name}')
        
        df_genre['taiwanTitle'] = ''
        df_genre['trailerLink'] = ''
        df_genre['posterPath'] = ''
        for idx, row in df_genre.iterrows():
            if row['averageRating'] >= percentile_90:
                tw_title, trailer, poster_path = get_tmdb_info(row['primaryTitle'], row['startYear'])
                df_genre.at[idx, 'taiwanTitle'] = tw_title
                df_genre.at[idx, 'trailerLink'] = trailer    
                df_genre.at[idx, 'posterPath'] = poster_path    
                print(f"Processed {row['primaryTitle']} ({row['startYear']}): {tw_title}, {trailer}")
        
        df_genre.to_excel(writer, sheet_name=genre_name[:31], index=False)


Processing genre: Empty
Processed Anaganaga (2025): Anaganaga Australia Lo, https://www.youtube.com/watch?v=-OuSOyjhots
Processed Gondhal (2025): Gondhal, https://www.youtube.com/watch?v=SwxwR37txRw
Processing genre: Sport
Processed Rocco and His Brothers (1960): 洛可兄弟, https://www.youtube.com/watch?v=54IuJ8ivZl0
Processed Rocky (1976): 洛基, https://www.youtube.com/watch?v=-Hk-LYcavrw
Processed Raging Bull (1980): 蠻牛, https://www.youtube.com/watch?v=G5RHRg6zEhY
Processed Hoop Dreams (1994): Hoop Dreams, https://www.youtube.com/watch?v=AglLHi4_0MM
Processed Children of Heaven (1997): 天堂的孩子, https://www.youtube.com/watch?v=dqxvZeQsVzY
Processed Lagaan: Once Upon a Time in India (2001): 榮耀之役, https://www.youtube.com/watch?v=rZPbpymefuE
Processed Million Dollar Baby (2004): 登峰造擊, https://www.youtube.com/watch?v=5_RsHRmIRBY
Processed Iqbal (2005): Iqbal, https://www.youtube.com/watch?v=VbEISCo8FKA
Processed Chak De! India (2007): 加油印度！, https://www.youtube.com/watch?v=6a0-dSMWm5g
Processed Th

In [8]:
# 讀取 Excel 檔案的所有 sheet 名稱
excel_path = "all_genres.xlsx"
sheets_dict = pd.read_excel(excel_path, sheet_name=None)  # 讀取所有 sheets

# 指定要排序的欄位名稱
sort_column = "averageRating"

# 建立一個新的 dict 來存放排序後的 DataFrame
sorted_sheets = {}

for sheet_name, df in sheets_dict.items():
    if sort_column in df.columns:
        sorted_df = df.sort_values(by=sort_column, ascending=False)
        sorted_sheets[sheet_name] = sorted_df
    else:
        print(f"Sheet '{sheet_name}' 中找不到欄位 '{sort_column}'，略過")

# 如果你想把結果寫回新的 Excel 檔案
with pd.ExcelWriter("movie recommendation.xlsx", engine='openpyxl') as writer:
    for sheet_name, sorted_df in sorted_sheets.items():
        sorted_df.to_excel(writer, sheet_name=sheet_name, index=False)
